

# Atlas-400F (notional cargo jet) — requirements-first program + report

Walkthrough: **Level-1** work is authored as composable **`Requirement`** packages (`model.requirement_package`). `L1MissionRequirements` demonstrates the **default package-level** `model.parameter`, `model.attribute`, and `model.constraint` surface, plus one deliberate **advanced leaf reqcheck** pattern (`requirement_input`, **`requirement_attribute`**, `requirement_accept_expr`) for the two atomic mission-closure checks in `examples/commercial_aircraft/program/`. The root `System` stays structural: scenario parameters, citations, allocations, and the composed **`Aircraft`** part. Roll-ups and external tools live on owned parts.

**Import path:** the example package is `thundergraph-model/examples/commercial_aircraft/`. The next cell adds the library root (for `tg_model`) and `examples/` to Python’s import path. After you change Python files in the example, restart the kernel and run from the top.


## What this demo is proving (two separate checks — read the report banner)

1. **Mission planning desk (illustrative):** a small external tool scales a **baseline maximum range** with payload and subtracts the **requested design range**, producing `mission_range_margin_m`. The constraint `mission_range_margin_non_negative` only checks that desk output. It is **not** a certification or compliance result.
2. **Declared envelope:** separate program constraints compare the **scenario** payload and range to **rolled-up masses** and **parameters** you set (for example maximum design range, maximum takeoff weight, maximum zero-fuel weight). This path does **not** use the desk’s physics.

A **PASS** in the summary means both checks passed for the chosen numbers — **not** that one unified physics model closed the mission.

**Level-1 table:** the **verification** column in the printed report is a short label for how each requirement is meant to be read: executable acceptance (mission closure with `allocate` inputs plus **`requirement_attribute`** margins on the requirement), shown by constraints, or context and citations only (regulatory framing — not a certification claim).

**Mission closure detail:** the mission **requirement package** holds **two** atomic Level-1 requirements (payload vs range, INCOSE-style). Each has its own **`requirement_input`** wiring, a **`requirement_attribute`** margin (`payload_margin_kg` or `range_margin_m`), and a **`requirement_accept_expr`** on that margin. After `instantiate`, those derived slots appear on **`ConfiguredModel.requirement_value_slots`** (see the short showcase cell below the path setup).

**Scope:** this notebook is a ThunderGraph API walkthrough (composition, `parameter_ref`, external compute in two places). It is not a full multi-discipline aircraft model.


## Composition (high level)

| Layer | Role |
|-------|------|
| `CargoJetProgram` | Structural root program: scenario parameters, citations, Level-1 requirement packages, allocations, and composed `aircraft` part |
| `Aircraft` | Vehicle part: assembly tree, operating empty weight roll-up, modeled maximum payload, mission desk output `mission_range_margin_m`, and envelope constraints |
| Major assemblies | Stub dry masses + wing assembly structural intensity (second external tool) |
| `requirement_attribute` (mission closure) | One derived margin per atomic mission requirement (payload closure vs range closure); materialized on `instantiate` as entries in `cm.requirement_value_slots` |


In [1]:
from __future__ import annotations

import sys
from pathlib import Path


def _thundergraph_pkg_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(24):
        if (p / "tg_model" / "__init__.py").is_file():
            return p
        nested = p / "thundergraph-model"
        if (nested / "tg_model" / "__init__.py").is_file():
            return nested
        if p.parent == p:
            break
        p = p.parent
    return start.resolve()


_cwd = Path.cwd().resolve()
_pkg_root = _thundergraph_pkg_root(_cwd)
_examples = _pkg_root / "examples"
for _p in (_pkg_root, _examples):
    _s = str(_p)
    if _s in sys.path:
        sys.path.remove(_s)
    sys.path.insert(0, _s)



## Level-1 mission closure: `requirement_input` + `requirement_attribute`

Composable **`Requirement`** types live in `program/l1_requirement_packages.py`. Mission closure is the deliberate **advanced leaf reqcheck** slice in this example: two requirements, **payload** (`req_cargo_design_mission_payload_closure`) and **range** (`req_cargo_design_mission_range_closure`). `CargoJetProgram` calls **`allocate`** twice on the aircraft part so each requirement only receives the inputs it needs. Each defines one **`requirement_attribute`** margin (envelope minus scenario) and a **`requirement_accept_expr`** that the margin is non-negative.

Run the next cell after `instantiate` to print the derived slot paths (also on `cm.requirement_value_slots`).

In [2]:
from commercial_aircraft import CargoJetProgram, reset_commercial_aircraft_types

reset_commercial_aircraft_types()
_cm = CargoJetProgram.instantiate()
print("requirement_attribute slots (derived on the requirement; see l1_requirement_packages.L1MissionRequirements):")
for _slot in _cm.requirement_value_slots:
    print(" ", _slot.path_string, f"({_slot.kind})")

requirement_attribute slots (derived on the requirement; see l1_requirement_packages.L1MissionRequirements):


## Nominal scenario (inside envelope, desk has margin)

Illustrative weight book: **140 t** operating empty weight, **240 t** maximum zero-fuel weight → **100 t** structural payload cap; **95 t** scenario payload. **8,000 km** design range versus **9,000 km** modeled maximum range and a **10,000 km** desk baseline so the illustrative margin stays positive.

**Run path:** **`ConfiguredModel.evaluate`** with **`ValueSlot`** keys (lazy compile; default validation). **`extract_cargo_jet_evaluation_report(cm, result)`** only needs the **`RunResult`**; pass **`ctx=…`** to that helper if you want optional **external-tool provenance** and **slot-state** counts (demo extras, not on **`RunResult`**). An optional **advanced** cell below shows `compile_graph` + `Evaluator`.

Inputs that move the verdict first: **`mission_desk_baseline_max_range_m`** (mission desk), **`scenario_design_range_m`**, **`modeled_max_design_range_m`** (envelope), and assembly dry masses (roll-up / maximum takeoff weight closure).


In [3]:
from unitflow import Quantity
from unitflow.catalogs.si import kg, m

from commercial_aircraft import CargoJetProgram, reset_commercial_aircraft_types
from commercial_aircraft.reporting import extract_cargo_jet_evaluation_report, format_cargo_jet_report


def run_case(label: str, make_inputs) -> None:
    reset_commercial_aircraft_types()
    cm = CargoJetProgram.instantiate()
    ac = cm.aircraft
    inputs = make_inputs(cm, ac)
    result = cm.evaluate(inputs=inputs)
    report = extract_cargo_jet_evaluation_report(cm, result)
    print(f"\n{'=' * 72}\nCASE: {label}\n{'=' * 72}")
    print(format_cargo_jet_report(report))
    print(f"\nEvaluator passed: {result.passed}")
    if result.failures:
        for line in result.failures:
            print("  failure:", line)


def nominal_inputs(cm, ac):
    return {
        cm.scenario_payload_mass_kg: Quantity(95_000, kg),
        cm.scenario_design_range_m: Quantity(8_000_000, m),
        cm.mission_desk_baseline_max_range_m: Quantity(10_000_000, m),
        cm.l1.mission.reserved_operational_buffer_kg: Quantity(0, kg),
        ac.scenario_payload_mass_kg: Quantity(95_000, kg),
        ac.scenario_design_range_m: Quantity(8_000_000, m),
        ac.modeled_max_design_range_m: Quantity(9_000_000, m),
        ac.notional_mzfw_kg: Quantity(240_000, kg),
        ac.notional_mtow_kg: Quantity(280_000, kg),
        ac.notional_trip_fuel_kg: Quantity(40_000, kg),
        ac.fuselage.dry_mass_kg: Quantity(45_000, kg),
        ac.wing.dry_mass_kg: Quantity(32_000, kg),
        ac.wing.notional_wing_span_m: Quantity(64, m),
        ac.empennage.dry_mass_kg: Quantity(8_000, kg),
        ac.landing_gear.dry_mass_kg: Quantity(12_000, kg),
        ac.propulsion_installation.dry_mass_kg: Quantity(28_000, kg),
        ac.aircraft_systems.dry_mass_kg: Quantity(15_000, kg),
    }


run_case("nominal (expect PASS)", nominal_inputs)



CASE: nominal (expect PASS)
Atlas-400F (notional) — evaluation snapshot
Verdict (desk margin + declared-envelope constraints + other checks): PASS
Thesis (read this once)
The demo uses two independent ideas: (1) **Mission desk** — a toy
external scaling model produces `mission_range_margin_m` and
`mission_range_margin_non_negative` checks it. (2) **Declared envelope**
— scenario payload/range are compared to rolled-up masses and parameters
(`notional_takeoff_mass_closure`, payload/range vs envelope). Passing
both is not a single physics closure; it is stitched verification for
teaching.
Mission desk track:
  Margin (raw): 2077236.6898835376 m
  Margin (rounded): ~2,077.2 km
  Margin ≥ 0: True
Declared envelope track (maximum takeoff weight / payload / range vs parameters):
  Envelope constraints all passed: True
  Evaluator completed without engine failures: True

Scenario inputs
quantity                          | value      | human (length)            
------------------------------

## Advanced (optional): explicit `compile_graph` + `Evaluator`

The same run can be written with the low-level pipeline: `compile_graph` → `validate_graph` → `Evaluator` with `inputs` keyed by `stable_id` strings. Use this when you need `evaluate_async`, custom graph inspection, or to match older examples. The default **`ConfiguredModel.evaluate`** path above is equivalent for synchronous runs.


In [4]:
# Equivalent explicit pipeline for the nominal inputs (same outcome as `evaluate` above).
from tg_model.execution.evaluator import Evaluator
from tg_model.execution.graph_compiler import compile_graph
from tg_model.execution.run_context import RunContext
from tg_model.execution.validation import validate_graph

reset_commercial_aircraft_types()
_cm_adv = CargoJetProgram.instantiate()
_ac_adv = _cm_adv.aircraft
_in = nominal_inputs(_cm_adv, _ac_adv)
_graph, _handlers = compile_graph(_cm_adv)
assert validate_graph(_graph, configured_model=_cm_adv).passed
_by_id = {slot.stable_id: val for slot, val in _in.items()}
_res = Evaluator(_graph, compute_handlers=_handlers).evaluate(RunContext(), inputs=_by_id)
print("Explicit pipeline passed:", _res.passed, "(should match nominal case above)")


Explicit pipeline passed: True (should match nominal case above)


## Stress case (desk fails; envelope may still pass)

Lower **`mission_desk_baseline_max_range_m`** to **6,000 km** while keeping an **8,000 km** requested range. The **desk margin** goes negative, so `mission_range_margin_non_negative` **fails**. The **envelope** constraints can still **pass**, which shows the two tracks are **not** the same check.


In [5]:
def stress_desk_inputs(cm, ac):
    out = nominal_inputs(cm, ac)
    out[cm.mission_desk_baseline_max_range_m] = Quantity(6_000_000, m)
    return out


run_case("stressed desk baseline (expect FAIL on desk track)", stress_desk_inputs)



CASE: stressed desk baseline (expect FAIL on desk track)
Atlas-400F (notional) — evaluation snapshot
Verdict (desk margin + declared-envelope constraints + other checks): FAIL
Thesis (read this once)
The demo uses two independent ideas: (1) **Mission desk** — a toy
external scaling model produces `mission_range_margin_m` and
`mission_range_margin_non_negative` checks it. (2) **Declared envelope**
— scenario payload/range are compared to rolled-up masses and parameters
(`notional_takeoff_mass_closure`, payload/range vs envelope). Passing
both is not a single physics closure; it is stitched verification for
teaching.
Mission desk track:
  Margin (raw): -1953657.9860698776 m
  Margin (rounded): ~-1,953.7 km
  Margin ≥ 0: False
Declared envelope track (maximum takeoff weight / payload / range vs parameters):
  Envelope constraints all passed: True
  Evaluator completed without engine failures: False

Scenario inputs
quantity                          | value     | human (length)          


---

## Behavioral model — flight-phase state machine + activity flows

`Aircraft` now carries two behavioral views that coexist with the structural/parametric model:

**State machine** — five high-level flight phases (FAA phase-of-flight taxonomy):

| Phase | Meaning |
|---|---|
| `parked` *(initial)* | Aircraft on ground, pre-departure or post-arrival |
| `pre_departure` | Clearance received; engines starting, taxi out |
| `airborne` | Rotation through gear retraction |
| `en_route` | Level cruise |
| `approach` | Descent, configuration, and landing roll |

Transitions fire on events: `clearance_received`, `takeoff_cleared`, `cruise_altitude_reached`, `top_of_descent`, `runway_vacated`, `diversion_declared` (any airborne phase → parked).

**Activity flows** (declared with `then=`):

- Pre-departure: `run_pre_flight_checks → start_engines → complete_taxi_out`
- Flight execution: `rotate_and_climb → retract_landing_gear → establish_cruise → initiate_descent → configure_for_approach → touchdown_and_rollout`

**Effect-only actions** (`record_dispatch_clearance`, `execute_go_around`) fire as transition side-effects and do **not** appear in the activity diagram — they are not part of any `then=` flow.

**Scenario** `nominal_departure_arrival` declares the expected event order and start/end state for one round trip, usable with `validate_scenario_trace` to check an execution trace against the authored contract.

The cells below use `behavior_authoring_projection`, `dispatch_event`, and `validate_scenario_trace` to exercise the full behavioral pipeline as a system test.

In [6]:
# ── System test 1: introspect declared behavioral structure ─────────────────
# behavior_authoring_projection is the tooling hook that extracts everything
# declared in define() for a given type. Used here to assert the Aircraft
# declares exactly what we expect before running any simulation.

from commercial_aircraft import CargoJetProgram, reset_commercial_aircraft_types
from tg_model.execution.behavior import behavior_authoring_projection

reset_commercial_aircraft_types()
cm = CargoJetProgram.instantiate()
aircraft_part = cm.path_registry["CargoJetProgram.aircraft"]
proj = behavior_authoring_projection(aircraft_part.definition_type)

expected_states = {"parked", "pre_departure", "airborne", "en_route", "approach"}
expected_flow_actions = {
    "run_pre_flight_checks", "start_engines", "complete_taxi_out",
    "rotate_and_climb", "retract_landing_gear", "establish_cruise",
    "initiate_descent", "configure_for_approach", "touchdown_and_rollout",
}
expected_effect_only = {"record_dispatch_clearance", "execute_go_around"}

assert set(proj["states"]) == expected_states, f"Unexpected states: {proj['states']}"
assert set(proj["actions"]) == expected_flow_actions | expected_effect_only
assert len(proj["transitions"]) == 8   # 5 mission-phase + 3 diversion paths
assert "nominal_departure_arrival" in proj["scenarios"]

# Verify then= chains are present in compiled metadata
nodes = aircraft_part.definition_type.compile()["nodes"]
assert nodes["run_pre_flight_checks"]["metadata"]["_then"] == "start_engines"
assert nodes["rotate_and_climb"]["metadata"]["_then"] == "retract_landing_gear"
assert nodes["configure_for_approach"]["metadata"]["_then"] == "touchdown_and_rollout"
assert "_then" not in nodes["execute_go_around"]["metadata"], "effect-only must not have _then"

print("✓ Structural assertions passed")
print(f"  States:      {sorted(proj['states'])}")
print(f"  Actions:     {sorted(proj['actions'])}")
print(f"  Transitions: {len(proj['transitions'])}")
print(f"  Scenarios:   {proj['scenarios']}")

✓ Structural assertions passed
  States:      ['airborne', 'approach', 'en_route', 'parked', 'pre_departure']
  Actions:     ['complete_taxi_out', 'configure_for_approach', 'establish_cruise', 'execute_go_around', 'initiate_descent', 'record_dispatch_clearance', 'retract_landing_gear', 'rotate_and_climb', 'run_pre_flight_checks', 'start_engines', 'touchdown_and_rollout']
  Transitions: 8
  Scenarios:   ['nominal_departure_arrival']


In [7]:
# ── System test 2: state machine dispatch — nominal round trip ───────────────
# Walk the aircraft through a full departure → cruise → arrival cycle and
# assert that the active state advances correctly at each step.

from tg_model.execution.behavior import (
    BehaviorTrace,
    DispatchOutcome,
    dispatch_event,
)
from tg_model.execution.run_context import RunContext

reset_commercial_aircraft_types()
cm = CargoJetProgram.instantiate()

# Bind the guard input to RunContext so weight_within_limits can read it.
# ctx.bind_input(stable_id, raw_float) — uses tg_model's slot machinery directly.
aircraft_part = cm.path_registry["CargoJetProgram.aircraft"]
ctx = RunContext()
ctx.bind_input(aircraft_part.scenario_payload_mass_kg.stable_id, 95_000.0)  # kg, raw float

trace = BehaviorTrace()

def step(event_name: str, expect: str) -> None:
    result = dispatch_event(ctx, aircraft_part, event_name, trace=trace)
    active = ctx.get_active_behavior_state(aircraft_part.path_string)
    assert result.outcome == DispatchOutcome.FIRED, (
        f"Event '{event_name}' did not fire (outcome={result.outcome}); "
        f"active state={active}"
    )
    assert active == expect, f"After '{event_name}': expected state '{expect}', got '{active}'"
    print(f"  {event_name:<30} → {active}")

print("Nominal round-trip state walk:")
step("clearance_received",      "pre_departure")
step("takeoff_cleared",         "airborne")
step("cruise_altitude_reached", "en_route")
step("top_of_descent",          "approach")
step("runway_vacated",          "parked")

print(f"\n✓ Final state: {ctx.get_active_behavior_state(aircraft_part.path_string)!r}")
print(f"  Transitions fired: {len(trace.steps)}")

Nominal round-trip state walk:
  clearance_received             → pre_departure
  takeoff_cleared                → airborne
  cruise_altitude_reached        → en_route
  top_of_descent                 → approach
  runway_vacated                 → parked

✓ Final state: 'parked'
  Transitions fired: 5


In [8]:
# ── System test 3: guard rejection — weight_within_limits ────────────────────
# If scenario_payload_mass_kg is 0 (or unbound), the weight_within_limits guard
# fails and takeoff_cleared must NOT fire.

from tg_model.execution.behavior import DispatchOutcome, dispatch_event
from tg_model.execution.run_context import RunContext

reset_commercial_aircraft_types()
cm_zero = CargoJetProgram.instantiate()
aircraft_zero = cm_zero.path_registry["CargoJetProgram.aircraft"]
ctx_zero = RunContext()
# Bind zero — guard checks mag > 0.0 so this should fail
ctx_zero.bind_input(aircraft_zero.scenario_payload_mass_kg.stable_id, 0.0)

# First, advance to pre_departure
r1 = dispatch_event(ctx_zero, aircraft_zero, "clearance_received")
assert r1.outcome == DispatchOutcome.FIRED

# Now try takeoff — guard should fail
r2 = dispatch_event(ctx_zero, aircraft_zero, "takeoff_cleared")
assert r2.outcome == DispatchOutcome.GUARD_FAILED, (
    f"Expected GUARD_FAILED, got {r2.outcome}"
)
still_pre_dep = ctx_zero.get_active_behavior_state(aircraft_zero.path_string)
assert still_pre_dep == "pre_departure"

print(f"✓ Guard correctly blocked takeoff — state remains '{still_pre_dep}'")

✓ Guard correctly blocked takeoff — state remains 'pre_departure'


In [9]:
# ── System test 4: diversion path ────────────────────────────────────────────
# Diversion can fire from airborne, en_route, or approach — all return to parked.

from tg_model.execution.behavior import DispatchOutcome, dispatch_event, BehaviorTrace
from tg_model.execution.run_context import RunContext

def diversion_from(phase_events: list) -> str:
    """Walk to a phase via phase_events, fire diversion_declared, return final state."""
    reset_commercial_aircraft_types()
    cm_d = CargoJetProgram.instantiate()
    ac_d = cm_d.path_registry["CargoJetProgram.aircraft"]
    ctx_d = RunContext()
    ctx_d.bind_input(ac_d.scenario_payload_mass_kg.stable_id, 95_000.0)
    for ev in phase_events:
        r = dispatch_event(ctx_d, ac_d, ev)
        assert r.outcome == DispatchOutcome.FIRED, f"Setup event '{ev}' didn't fire"
    r_div = dispatch_event(ctx_d, ac_d, "diversion_declared")
    assert r_div.outcome == DispatchOutcome.FIRED
    return ctx_d.get_active_behavior_state(ac_d.path_string)

print("Diversion from airborne  →", diversion_from(["clearance_received", "takeoff_cleared"]))
print("Diversion from en_route  →", diversion_from(["clearance_received", "takeoff_cleared", "cruise_altitude_reached"]))
print("Diversion from approach  →", diversion_from(["clearance_received", "takeoff_cleared", "cruise_altitude_reached", "top_of_descent"]))

for phase_events in [
    ["clearance_received", "takeoff_cleared"],
    ["clearance_received", "takeoff_cleared", "cruise_altitude_reached"],
    ["clearance_received", "takeoff_cleared", "cruise_altitude_reached", "top_of_descent"],
]:
    assert diversion_from(phase_events) == "parked"

print("\n✓ All diversion paths correctly return to 'parked'")

Diversion from airborne  → parked
Diversion from en_route  → parked
Diversion from approach  → parked

✓ All diversion paths correctly return to 'parked'


In [10]:
# ── System test 5: scenario validation ───────────────────────────────────────
# validate_scenario_trace checks the trace recorded during dispatch_event calls
# against the authored scenario contract (expected_event_order, initial/final state).

from tg_model.execution.behavior import (
    BehaviorTrace,
    DispatchOutcome,
    dispatch_event,
    validate_scenario_trace,
)
from tg_model.execution.run_context import RunContext

reset_commercial_aircraft_types()
cm_scen = CargoJetProgram.instantiate()
ac_scen = cm_scen.path_registry["CargoJetProgram.aircraft"]
ctx_scen = RunContext()
ctx_scen.bind_input(ac_scen.scenario_payload_mass_kg.stable_id, 95_000.0)

trace_scen = BehaviorTrace()
for ev in ["clearance_received", "takeoff_cleared", "cruise_altitude_reached",
           "top_of_descent", "runway_vacated"]:
    result = dispatch_event(ctx_scen, ac_scen, ev, trace=trace_scen)
    assert result.outcome == DispatchOutcome.FIRED, f"Event '{ev}' failed"

ok, errors = validate_scenario_trace(
    definition_type=ac_scen.definition_type,
    scenario_name="nominal_departure_arrival",
    part_path=ac_scen.path_string,
    trace=trace_scen,
    ctx=ctx_scen,
)

if ok:
    print("✓ Scenario 'nominal_departure_arrival' validated successfully")
else:
    print("✗ Scenario validation failed:")
    for e in errors:
        print("  -", e)
    raise AssertionError("Scenario validation failed")

print(f"  Events fired:  {[s.event_name for s in trace_scen.steps]}")
print(f"  Final state:   {ctx_scen.get_active_behavior_state(ac_scen.path_string)!r}")

✓ Scenario 'nominal_departure_arrival' validated successfully
  Events fired:  ['clearance_received', 'takeoff_cleared', 'cruise_altitude_reached', 'top_of_descent', 'runway_vacated']
  Final state:   'parked'


In [11]:
# ── System test 6: bundle_walker behavioral projection ───────────────────────
# The projection pipeline (bundle_walker._behavioral_records) is the bridge
# between the DSL and the ThunderGraph graph database. This test is only
# meaningful inside the backend monorepo where neomodel is available.
# When running this notebook standalone from the tg-model package, the import
# will be skipped gracefully.

try:
    from common.services.tg_model_projector.bundle_walker import _behavioral_records
    _backend_available = True
except ImportError:
    _backend_available = False
    print("ℹ  Skipping Test 6: backend projector (common.services) not in scope.")
    print("   Run this test from the backend-monorepo virtualenv to verify Neo4j projection.")

if _backend_available:
    reset_commercial_aircraft_types()
    cm_proj = CargoJetProgram.instantiate()
    ac_proj = cm_proj.path_registry["CargoJetProgram.aircraft"]

    beh = _behavioral_records(ac_proj)

    assert len(beh["states"]) == 5
    initial_states = [s for s in beh["states"] if s["is_initial"]]
    assert len(initial_states) == 1 and initial_states[0]["name"] == "parked"

    assert len(beh["transitions"]) == 8
    triggers = {t["trigger"] for t in beh["transitions"]}
    assert "takeoff_cleared" in triggers
    assert "diversion_declared" in triggers

    action_names = {a["name"] for a in beh["actions"]}
    assert "rotate_and_climb" in action_names
    assert "touchdown_and_rollout" in action_names
    assert "execute_go_around" not in action_names,       "effect-only must be excluded"
    assert "record_dispatch_clearance" not in action_names, "effect-only must be excluded"

    assert len(beh["successions"]) == 7  # 2 pre-dep + 5 flight execution then= chains

    print(f"✓ bundle_walker behavioral records verified")
    print(f"  States:      {sorted(s['name'] for s in beh['states'])}")
    print(f"  Transitions: {len(beh['transitions'])} (triggers: {sorted(triggers)})")
    print(f"  Actions:     {sorted(action_names)}")
    print(f"  Successions: {len(beh['successions'])}")

ℹ  Skipping Test 6: backend projector (common.services) not in scope.
   Run this test from the backend-monorepo virtualenv to verify Neo4j projection.
